# Выбор маски для усложнённых задач

Сравнение двух рейтингов голов на простой задаче: сколько голов надо отключить по каждому, чтобы получить один и тот же ущерб. Нужна пара с равным эффектом: тогда на усложнённых задачах разницу рейтингов можно отделить от разницы в силе маски.

Метрики ответа, от строгой к мягкой:

- **точная фраза** — эталон стоит в ответе дословно и подряд, без учёта регистра и пунктуации;
- **ROUGE-3 и ROUGE-2 recall** — доля троек и пар соседних слов эталона, которые есть в ответе;
- **ROUGE-1 recall** — доля отдельных слов эталона в ответе;
- **успех** — ROUGE-1 recall выше 50, критерий статьи.

ROUGE-1 засчитывает общие слова и в ответах, где игла не найдена; пары и тройки слов случайно почти не совпадают, поэтому ROUGE-2 и ROUGE-3 ближе к точной фразе.

Источники: наш перебор масок (`mask_name`), наши прогоны с маской по массе внимания (`mass_name`) и прогоны ветки `c0` по тому же рейтингу массы (`external`). Модель и GPU не нужны; pandas не используется.

In [ ]:
import glob
import json
import os
import re
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

## Настройки

`copy_heads` — число голов по копированию, под которое подбирается число голов по массе.

In [ ]:
mask_name = 'qwen3_mask'                    # наш перебор: none, top*, random*, groups*
mass_name = 'qwen3_mask_attention_mass_2'   # наши прогоны с маской по массе внимания: top*
mass_key = 'attention_mass'                 # по этой подстроке в имени файла рейтинга прогон относится к массе
external = 'data/mask_runs/c0_mass_top5_40/cases.jsonl'   # прогоны c0, путь от корня репозитория; None — нет
copy_heads = 60

# ноутбук может лежать в source/viz или запускаться из source либо из корня репозитория
root = next(p for p in ('../../results/new', '../results/new', 'results/new') if os.path.isdir(p))
repo = os.path.dirname(os.path.dirname(os.path.normpath(root)))
out_dir = 'viz' if os.path.isdir('viz') else '.'

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 220, 'font.size': 11})

# источник -> (подпись, цвет, стиль линии)
sources = {'copy': ('копирование, наш прогон', '#eb6834', '-'), 'mass': ('масса внимания, наш прогон', '#8a4fd3', '-'),
           'mass_c0': ('масса внимания, прогон c0', '#8a4fd3', '--'), 'random': ('случайные россыпью', '#2a78d6', '-'),
           'groups': ('случайные группами', '#1baf7a', '-')}
# метрика -> (подпись, множитель)
metrics = {'exact': ('Точная фраза, %', 100), 'rouge3': ('Средний ROUGE-3 recall', 1), 'rouge2': ('Средний ROUGE-2 recall', 1),
           'rouge': ('Средний ROUGE-1 recall', 1), 'success': ('ROUGE-1 recall > 50, %', 100)}
score_cmap = LinearSegmentedColormap.from_list('score', ['#f0efec', '#9ec5f4', '#2a78d6', '#0d366b'])
grid_color, ink = '#e6e5e1', '#52514e'

def style(ax):
    ax.grid(axis='y', color=grid_color, linewidth=1)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)

## Загрузка

Точная фраза считается здесь же по тексту ответа, если в файле прогона её ещё нет. ROUGE-2 и ROUGE-3 берутся из файла прогона, а для старых прогонов считаются библиотекой `rouge_score`; если её в окружении ноутбука нет, эти две метрики для старых прогонов пропускаются (тогда сначала пересчитайте прогоны: `rh.metrics --rescore`). Прогон без маски — общая нулевая точка всех наших линий; у прогонов `c0` своя.

In [ ]:
def plain(text):
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', ' ', unicodedata.normalize('NFKC', text).lower())).strip()

def exact_phrase(reference, response):
    return f' {plain(reference)} ' in f' {plain(response)} '

try:
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(['rouge2', 'rouge3'], use_stemmer=True)
except ImportError:
    scorer = None

def ngram_recall(s):
    if 'rouge2_recall' in s and 'rouge3_recall' in s:
        return s['rouge2_recall'], s['rouge3_recall']
    if scorer is None:
        return float('nan'), float('nan')
    score = scorer.score(s['reference'], s['response'])
    return score['rouge2'].recall * 100, score['rouge3'].recall * 100

def row(source, heads, s):
    rouge2, rouge3 = ngram_recall(s)
    return {'source': source, 'heads': heads, 'length': s['context_length'], 'depth': s['depth_percent'], 'rouge': s['rouge1_recall'],
            'rouge2': rouge2, 'rouge3': rouge3, 'success': float(s['success']),
            'exact': float(s['exact'] if 'exact' in s else exact_phrase(s['reference'], s['response']))}

def read(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

rows = []
kinds = {'top': 'copy', 'random': 'random', 'random_groups': 'groups'}
for folder, own in [(f, True) for f in sorted(glob.glob(f'{root}/{mask_name}/*/'))] + [(f, False) for f in sorted(glob.glob(f'{root}/{mass_name}/*/'))]:
    if not os.path.exists(f'{folder}/samples.jsonl'):
        continue
    run = json.load(open(f'{folder}/spool/run.json')) if os.path.exists(f'{folder}/spool/run.json') else {}
    kind, heads = run.get('mask', 'none'), len(run.get('block_list') or [])
    by_mass = kind == 'top' and mass_key in os.path.basename((run.get('args') or {}).get('mask_file') or '')
    if not own and not by_mass:
        continue   # из папки второго рейтинга нужны только прогоны с его лучшими головами
    samples = read(f'{folder}/samples.jsonl')
    if kind == 'none':
        for source in ('copy', 'mass', 'random', 'groups'):
            rows += [row(source, 0, s) for s in samples]
    else:
        rows += [row('mass' if by_mass else kinds[kind], heads, s) for s in samples]
if external and os.path.exists(os.path.join(repo, external)):
    rows += [row('mass_c0', s['heads'], s) for s in read(os.path.join(repo, external))]
elif external:
    print('нет файла прогонов c0:', os.path.join(repo, external))

def curve(source, metric):
    """[(число голов, среднее значение метрики)] по возрастанию числа голов."""
    values = {}
    for r in rows:
        if r['source'] == source:
            values.setdefault(r['heads'], []).append(r[metric])
    return [(k, float(np.mean(v)) * metrics[metric][1]) for k, v in sorted(values.items()) if not np.isnan(v).any()]

present = [s for s in sources if curve(s, 'rouge')]
if not curve('copy', 'rouge2'):
    print('ROUGE-2 и ROUGE-3 для наших прогонов не посчитаны: нет rouge_score и нет этих полей в samples.jsonl')
print(f'примеров: {len(rows)}; источники: ' + ', '.join(f"{sources[s][0]} ({len(curve(s, 'rouge'))} точек)" for s in present))

## Таблица

По строке на источник, по столбцу на число отключённых голов.

In [ ]:
all_heads = sorted({r['heads'] for r in rows})
for metric, (label, _) in metrics.items():
    print(f'\n{label}')
    print(f"{'':<28}" + ''.join(f'{k:>7}' for k in all_heads))
    for source in present:
        points = dict(curve(source, metric))
        print(f'{sources[source][0]:<28}' + ''.join(f'{points[k]:>7.1f}' if k in points else f"{'·':>7}" for k in all_heads))

## Качество от числа отключённых голов

Главный график: все метрики рядом, от строгой к мягкой. Горизонтальный пунктир — уровень выбранной маски по копированию (`copy_heads`); число голов по массе с тем же ущербом — там, где фиолетовая линия пересекает пунктир.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.ravel()
for ax in axes[len(metrics):]:
    ax.set_visible(False)
for ax, (metric, (label, _)) in zip(axes, metrics.items()):
    for source in present:
        points = curve(source, metric)
        name, color, line = sources[source]
        ax.plot([k for k, _ in points], [v for _, v in points], color=color, linestyle=line, linewidth=2, marker='o', markersize=5, label=name)
    target = dict(curve('copy', metric)).get(copy_heads)
    if target is not None:
        ax.axhline(target, color=ink, linestyle=':', linewidth=1.2)
        ax.annotate(f'копирование, {copy_heads} голов: {target:.1f}', xy=(1, target), xycoords=('axes fraction', 'data'),
                    ha='right', va='bottom', fontsize=10, color=ink)
    ax.set_ylim(0, 105)
    ax.set_xlabel('Отключено голов')
    ax.set_title(label)
    style(ax)
axes[0].legend(frameon=False, loc='lower left')
fig.suptitle('Два рейтинга голов на простой задаче')
fig.tight_layout()
fig.savefig(f'{out_dir}/mask_choice_by_heads.png', bbox_inches='tight')
plt.show()

## Начало кривых крупно

Рейтинг по массе ломает поиск иглы уже на 15–20 головах, поэтому тот же график по точной фразе до 80 голов.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for source in present:
    points = [(k, v) for k, v in curve(source, 'exact') if k <= 80]
    name, color, line = sources[source]
    ax.plot([k for k, _ in points], [v for _, v in points], color=color, linestyle=line, linewidth=2, marker='o', markersize=6, label=name)
    if source in ('copy', 'mass', 'mass_c0'):
        for k, v in points:
            ax.annotate(f'{v:.0f}', xy=(k, v), xytext=(0, 7), textcoords='offset points', ha='center', fontsize=9, color=color)
ax.set_ylim(0, 110)
ax.set_xlabel('Отключено голов')
ax.set_ylabel(metrics['exact'][0])
ax.legend(frameon=False, loc='lower left')
style(ax)
fig.tight_layout()
fig.savefig(f'{out_dir}/mask_choice_exact.png', bbox_inches='tight')
plt.show()

## Сколько голов по массе дают тот же ущерб

Для каждой метрики берётся значение маски по копированию на `copy_heads` головах и ищется точка на кривой массы с тем же значением (линейная интерполяция между соседними прогонами). Кривая массы: наши прогоны, а где их нет — прогоны `c0`.

Если по разным метрикам получаются разные числа голов, верить стоит строгим — точной фразе, ROUGE-3 и ROUGE-2: ROUGE-1 засчитывает общие слова в ответах, где игла не найдена.

In [ ]:
def heads_for(points, value):
    """Число голов, при котором кривая проходит через value; None, если не проходит."""
    for (k0, v0), (k1, v1) in zip(points, points[1:]):
        if v0 != v1 and min(v0, v1) <= value <= max(v0, v1):
            return k0 + (k1 - k0) * (v0 - value) / (v0 - v1)
    return None

def mass_curve(metric):
    return sorted({**dict(curve('mass_c0', metric)), **dict(curve('mass', metric))}.items())

matched = {}
print(f"{'метрика':<26} {'копирование, ' + str(copy_heads):>18} {'масса, голов':>14}   соседние прогоны по массе")
# метрика может быть не посчитана для наших прогонов (ROUGE-2/3 без rouge_score): тогда строка говорит, что прогона нет
for metric, (label, _) in metrics.items():
    target = dict(curve('copy', metric)).get(copy_heads)
    if target is None:
        print(f'{label:<26} нет прогона по копированию на {copy_heads} головах')
        continue
    points = mass_curve(metric)
    matched[metric] = heads_for(points, target)
    near = ', '.join(f'{k}: {v:.1f}' for k, v in points if matched[metric] is not None and abs(k - matched[metric]) <= 6)
    found = f'{matched[metric]:.1f}' if matched[metric] is not None else 'нет'
    print(f'{label:<26} {target:>18.1f} {found:>14}   {near}')

if matched.get('exact') is not None:
    print(f"\nПара с равной долей точных ответов: {copy_heads} голов по копированию и около {matched['exact']:.0f} по массе.")
    controls = [(s, dict(curve(s, 'exact')).get(copy_heads)) for s in ('random', 'groups')]
    print('Контроль на том же числе голов, точная фраза: ' + ', '.join(f'{sources[s][0]} {v:.1f}' for s, v in controls if v is not None))

## Карты «длина × глубина» для выбранных масок

Точная фраза по клеткам сетки: маска по копированию на `copy_heads` головах, два ближайших прогона по массе вокруг найденного числа голов и случайные головы. Показывает, одинаково ли две маски распределяют ущерб по длинам и глубинам, а не только в среднем.

In [ ]:
def nearest(source, value, count=1):
    ks = sorted({r['heads'] for r in rows if r['source'] == source and r['heads'] > 0}, key=lambda k: abs(k - value))
    return sorted(ks[:count])

center = matched.get('exact') or 20
mass_source = lambda k: 'mass' if k in dict(curve('mass', 'exact')) else 'mass_c0'
chosen = [('copy', copy_heads)] + [(mass_source(k), k) for k in nearest('mass', center, 2) or nearest('mass_c0', center, 2)]
if not any(s == 'mass_c0' for s, _ in chosen):
    chosen += [('mass_c0', k) for k in nearest('mass_c0', center, 1)]
chosen += [('random', copy_heads)]
chosen = [(s, k) for s, k in chosen if any(r['source'] == s and r['heads'] == k for r in rows)]

lengths, depths = sorted({r['length'] for r in rows}), sorted({r['depth'] for r in rows})
map_cmap = score_cmap.copy()
map_cmap.set_bad('#ffffff')
fig, axes = plt.subplots(1, len(chosen), figsize=(7 * len(chosen), 4.6), squeeze=False)
for ax, (source, k) in zip(axes[0], chosen):
    cell = {}
    for r in rows:
        if r['source'] == source and r['heads'] == k:
            cell.setdefault((r['depth'], r['length']), []).append(r['exact'])
    table = np.array([[100 * np.mean(cell[(d, l)]) if (d, l) in cell else np.nan for l in lengths] for d in depths])
    image = ax.imshow(np.ma.masked_invalid(table), aspect='auto', cmap=map_cmap, vmin=0, vmax=100)
    ax.set_xticks(range(0, len(lengths), 2), lengths[::2], rotation=45)
    ax.set_yticks(range(len(depths)), depths)
    ax.set_xlabel('Длина контекста, токенов')
    ax.set_title(f'{sources[source][0]}, {k} голов: {np.nanmean(table):.1f}', fontsize=12)
axes[0, 0].set_ylabel('Глубина иглы, %')
fig.colorbar(image, ax=axes, label=metrics['exact'][0], fraction=0.015, pad=0.015)
fig.savefig(f'{out_dir}/mask_choice_maps.png', bbox_inches='tight')
plt.show()

## По глубине иглы

Точная фраза по глубинам для тех же масок, среднее по длинам. Если две маски уравнены в среднем, но одна бьёт по середине контекста, а другая по краям, это видно здесь.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for source, k in chosen:
    line = {}
    for r in rows:
        if r['source'] == source and r['heads'] == k:
            line.setdefault(r['depth'], []).append(r['exact'])
    name, color, dash = sources[source]
    xs = sorted(line)
    # у двух прогонов по массе один цвет: второй рисуется бледнее
    faint = source.startswith('mass') and any(s.startswith('mass') and kk < k for s, kk in chosen)
    ax.plot(xs, [100 * np.mean(line[x]) for x in xs], linewidth=2, marker='o', markersize=5, linestyle=dash, color=color,
            alpha=0.5 if faint else 1.0, label=f'{name}, {k} голов')
ax.set_ylim(0, 105)
ax.set_xlabel('Глубина иглы, % контекста')
ax.set_ylabel(metrics['exact'][0])
ax.legend(frameon=False, loc='center left', bbox_to_anchor=(1.0, 0.5))
style(ax)
fig.tight_layout()
fig.savefig(f'{out_dir}/mask_choice_by_depth.png', bbox_inches='tight')
plt.show()